# Laboratorio 4: análisis de reseñas con Hugging Face
**Alcance:** dataset y partes A, B, C y D.

Este notebook construye un flujo de análisis de sentimientos y reconocimiento de entidades nombradas (NER) en español. Contiene 30 reseñas simuladas, procesa ambos modelos en lote, integra las predicciones y prepara estadísticas y archivos CSV.

**Uso en Google Colab:** sube este archivo y ejecuta las celdas en orden. La primera ejecución necesita conexión para instalar dependencias y descargar los modelos. Puede funcionar con CPU; si hay una GPU disponible, la selecciona automáticamente.

El notebook se entrega **sin ejecutar**, sin predicciones ni estadísticas precalculadas. Los archivos de resultados se crean al ejecutar las celdas de exportación.

## 1. Instalación y configuración
Se utiliza la API de pipelines de Transformers 4.x. PyTorch 2.6 o posterior permite cargar los pesos del modelo NER publicados en formato PyTorch. Si Colab solicita reiniciar la sesión después de instalar, reiníciala antes de continuar con las importaciones.

In [ ]:
%pip install -q "transformers>=4.44,<5" "torch>=2.6" pandas

In [ ]:
import json

import pandas as pd
import torch
import transformers
from IPython.display import display
from transformers import AutoTokenizer, pipeline

MODELO_SENTIMIENTO = "pysentimiento/robertuito-sentiment-analysis"
MODELO_NER = "mrm8488/bert-spanish-cased-finetuned-ner"
BATCH_SIZE = 8
DEVICE = 0 if torch.cuda.is_available() else -1

pd.set_option("display.max_colwidth", None)

print("Dispositivo:", "GPU" if DEVICE == 0 else "CPU")
print("Transformers:", transformers.__version__)
print("PyTorch:", torch.__version__)
print("Pandas:", pd.__version__)

## 2. Dataset: 30 reseñas inventadas
Hay cinco reseñas por cada grupo solicitado en el paso 4: positivas, negativas, mixtas, con negación y con posibles entidades. Esos cinco grupos suman 25; se añaden cinco reseñas neutrales para cumplir el mínimo de 30.

Las reseñas fueron generadas con asistencia de IA para este laboratorio. Las personas, organizaciones y localidades mencionadas son ficticias y no representan experiencias reales ni datos personales reales. Cada reseña del grupo de entidades menciona una persona, una organización y un lugar.

La columna `categoria_ejemplo` describe cómo se diseñó el texto; **no es una etiqueta de referencia validada** ni se entrega al modelo. Un texto puede presentar varias características, aunque aquí tenga un único grupo principal.

In [ ]:
ejemplos = [
    # Cinco reseñas positivas.
    ("positivo", "El café estaba delicioso y el pan recién horneado. Salí muy satisfecho con la compra."),
    ("positivo", "Los audífonos tienen un sonido excelente y son muy cómodos. Los recomiendo totalmente."),
    ("positivo", "La habitación estaba impecable y la cama era comodísima. Pasamos una estancia maravillosa."),
    ("positivo", "Mi pedido llegó antes de lo previsto y todo venía perfectamente protegido. Excelente servicio."),
    ("positivo", "La aplicación es fácil de usar y me permitió comprar en pocos minutos. Me encantó la experiencia."),

    # Cinco reseñas negativas.
    ("negativo", "La pizza llegó fría, quemada y con ingredientes equivocados. Fue una compra pésima."),
    ("negativo", "Los zapatos se rompieron después de dos días de uso. La calidad es terrible."),
    ("negativo", "Esperé una hora para recibir una respuesta grosera. El servicio fue decepcionante."),
    ("negativo", "La habitación estaba sucia y olía a humedad. La estancia fue desagradable."),
    ("negativo", "La aplicación se cierra cada vez que intento pagar. Comprar aquí resulta frustrante."),

    # Cinco reseñas mixtas.
    ("mixto", "La comida estaba deliciosa, pero tardaron demasiado en servirla."),
    ("mixto", "La cámara toma fotos excelentes, aunque la batería dura muy poco."),
    ("mixto", "El hotel tiene una vista hermosa, pero el ruido de la calle impide descansar."),
    ("mixto", "El paquete llegó rápido, aunque el producto tenía varios rayones."),
    ("mixto", "El personal fue amable y resolvió mis dudas, pero el precio final me pareció excesivo."),

    # Cinco reseñas con negación.
    ("negacion", "No me gustó la sopa: estaba demasiado salada y apenas pude terminarla."),
    ("negacion", "No tengo ninguna queja; el pedido llegó completo y en perfecto estado."),
    ("negacion", "El producto no es malo, aunque esperaba una mejor presentación."),
    ("negacion", "Nunca había recibido una atención tan buena. Volveré con mucho gusto."),
    ("negacion", "No volvería a comprar aquí, ni siquiera con un descuento."),

    # Cinco reseñas con posibles entidades: todos los nombres son inventados.
    ("entidades", "La asesora Lucía Brisaverde me atendió muy bien en Tiendas Nube Clara, en Villa Luminaria."),
    ("entidades", "Mateo Soldebrisa, de Mensajería Faro Violeta, entregó mi paquete roto en Puerto Lunaclara. Quedé muy molesto."),
    ("entidades", "En el Hotel Jardín de Cobre, ubicado en Valle Nacarino, Elena Marazul fue amable, pero la habitación estaba sucia."),
    ("entidades", "Tomás Cielomanso preparó un café delicioso en Cafetería Cometa Azul, en Ciudad Alboral. Recomiendo la visita."),
    ("entidades", "Sara Ventoluna, de Librería Bosque de Papel, confirmó que mi pedido estará disponible mañana en San Olmo de Bruma."),

    # Cinco reseñas neutrales adicionales para alcanzar el mínimo de 30.
    ("neutral_adicional", "El paquete contiene dos cuadernos y un lápiz."),
    ("neutral_adicional", "Compré la versión de quinientos mililitros y elegí el envase azul."),
    ("neutral_adicional", "El recibo indica que la compra se realizó el martes por la tarde."),
    ("neutral_adicional", "La tienda ofrece retiro en sucursal y entrega a domicilio."),
    ("neutral_adicional", "El manual viene en español y se encuentra dentro de la caja."),
]

dataset = pd.DataFrame(
    [
        {"review_id": indice, "categoria_ejemplo": categoria, "text": texto}
        for indice, (categoria, texto) in enumerate(ejemplos, start=1)
    ]
)

assert len(dataset) == 30
assert dataset["review_id"].is_unique
assert dataset["text"].is_unique
assert dataset["text"].str.strip().ne("").all()
assert dataset["categoria_ejemplo"].value_counts().eq(5).all()

reseñas = dataset["text"].tolist()
display(dataset)
display(
    dataset.groupby("categoria_ejemplo")
    .size()
    .rename("cantidad")
    .reset_index()
)

## Parte A. Análisis de sentimientos
Se emplea [pysentimiento/robertuito-sentiment-analysis](https://huggingface.co/pysentimiento/robertuito-sentiment-analysis), entrenado para sentimiento en español. Se conservan sus etiquetas originales: `POS` (positivo), `NEG` (negativo) y `NEU` (neutral).

El modelo asigna una sola etiqueta por reseña; no tiene una clase específica para opiniones mixtas. El score corresponde a la etiqueta elegida y no es una medida de exactitud ni una garantía de acierto. Se conserva el texto original y se limita la entrada de sentimiento a 128 tokens, conforme a la configuración de RoBERTuito.

In [ ]:
sentiment = pipeline(
    "sentiment-analysis",
    model=MODELO_SENTIMIENTO,
    tokenizer=MODELO_SENTIMIENTO,
    device=DEVICE,
)

print("Etiquetas de sentimiento del modelo:", sentiment.model.config.id2label)

resultados_sentimiento = sentiment(
    reseñas,
    batch_size=BATCH_SIZE,
    truncation=True,
    max_length=128,
)

assert len(resultados_sentimiento) == len(dataset)

df_sentimiento = dataset[["review_id", "text"]].copy()
df_sentimiento["sentiment"] = [
    resultado["label"] for resultado in resultados_sentimiento
]
df_sentimiento["sentiment_score"] = [
    float(resultado["score"]) for resultado in resultados_sentimiento
]

display(df_sentimiento)

## Parte B. Reconocimiento de entidades nombradas (NER)
Se utiliza [mrm8488/bert-spanish-cased-finetuned-ner](https://huggingface.co/mrm8488/bert-spanish-cased-finetuned-ner), un modelo BETO ajustado para NER en español. Es un segundo modelo porque sentimiento y reconocimiento de entidades son tareas diferentes.

La opción `aggregation_strategy="simple"` agrupa tokens de una misma entidad. Se consultan las etiquetas en la configuración y se conserva `entity_group` de cada predicción, sin imponer una lista de categorías. Los tipos reportados en la parte D serán únicamente los encontrados en estas reseñas.

Para guardar el texto exacto de cada mención se usan sus posiciones en la reseña original. Una reseña sin entidades conserva una lista vacía.

In [ ]:
tokenizer_ner = AutoTokenizer.from_pretrained(MODELO_NER, use_fast=True)

ner = pipeline(
    "token-classification",
    model=MODELO_NER,
    tokenizer=tokenizer_ner,
    aggregation_strategy="simple",
    device=DEVICE,
)

print("Etiquetas NER del modelo:", ner.model.config.id2label)

# Las reseñas son breves. Esta comprobación evita omitir texto por truncamiento.
longitudes_ner = [
    len(tokenizer_ner(texto, add_special_tokens=True)["input_ids"])
    for texto in reseñas
]
assert max(longitudes_ner) <= ner.model.config.max_position_embeddings, (
    "Hay una reseña demasiado larga para NER; divídela antes de procesarla."
)

resultados_ner = ner(reseñas, batch_size=BATCH_SIZE)
assert len(resultados_ner) == len(dataset)

entidades_por_reseña = [
    [
        {
            "label": entidad["entity_group"],
            "text": texto[int(entidad["start"]):int(entidad["end"])],
            "score": float(entidad["score"]),
        }
        for entidad in entidades
    ]
    for texto, entidades in zip(reseñas, resultados_ner)
]

filas_entidades = [
    {"review_id": int(review_id), **entidad}
    for review_id, entidades in zip(dataset["review_id"], entidades_por_reseña)
    for entidad in entidades
]
df_entidades = pd.DataFrame(
    filas_entidades,
    columns=["review_id", "label", "text", "score"],
)

display(df_entidades)

## Parte C. Flujo integrado
Se combinan las predicciones ya calculadas, manteniendo el identificador y el orden del dataset. Cada registro contiene texto, etiqueta, score de sentimiento y una lista de entidades con su tipo, texto y score. También se guarda la categoría de diseño para poder revisar el dataset.

`resultados_integrados` es la estructura de diccionarios y `df` es su representación tabular. La cantidad de entidades cuenta menciones: si un nombre aparece dos veces y el modelo detecta ambas, se cuentan dos.

In [ ]:
resultados_integrados = [
    {
        "review_id": int(fila.review_id),
        "text": fila.text,
        "categoria_ejemplo": fila.categoria_ejemplo,
        "sentiment": resultado_sentimiento["label"],
        "sentiment_score": float(resultado_sentimiento["score"]),
        "entities": entidades,
    }
    for fila, resultado_sentimiento, entidades in zip(
        dataset.itertuples(index=False),
        resultados_sentimiento,
        entidades_por_reseña,
    )
]

df = pd.DataFrame(resultados_integrados)
df["entity_count"] = df["entities"].apply(len)

assert len(df) == len(dataset)
assert df["sentiment_score"].between(0, 1).all()

# Muestra la estructura integrada de la primera reseña del grupo de entidades.
indice_ejemplo = dataset.index[dataset["categoria_ejemplo"].eq("entidades")][0]
print(json.dumps(resultados_integrados[indice_ejemplo], ensure_ascii=False, indent=2))
display(df)

## Parte D. Estadísticas y exportación
Las siguientes celdas reportan la cantidad de reseñas por etiqueta, el score promedio global y por etiqueta, las cinco reseñas de menor confianza, el promedio de entidades por reseña y los tipos de entidad encontrados.

El promedio de entidades incluye las reseñas sin entidades. Las etiquetas de sentimiento disponibles que no aparezcan se muestran con conteo cero; su promedio por etiqueta queda sin valor. Los scores de sentimiento y NER se mantienen separados.

In [ ]:
etiquetas_sentimiento = list(dict.fromkeys(sentiment.model.config.id2label.values()))

resumen_sentimiento = (
    df.groupby("sentiment")
    .agg(
        cantidad_reseñas=("review_id", "size"),
        score_promedio=("sentiment_score", "mean"),
    )
    .reindex(etiquetas_sentimiento)
    .rename_axis("sentiment")
    .reset_index()
)
resumen_sentimiento["cantidad_reseñas"] = (
    resumen_sentimiento["cantidad_reseñas"].fillna(0).astype(int)
)

score_promedio = float(df["sentiment_score"].mean())
promedio_entidades = float(df["entity_count"].mean())
tipos_encontrados = sorted(df_entidades["label"].unique().tolist())

menor_confianza = (
    df.sort_values(["sentiment_score", "review_id"])
    .head(5)[
        ["review_id", "text", "categoria_ejemplo", "sentiment", "sentiment_score"]
    ]
    .reset_index(drop=True)
)

conteo_tipos_entidad = (
    df_entidades.groupby("label")
    .size()
    .rename("cantidad_menciones")
    .reset_index()
    .sort_values("cantidad_menciones", ascending=False)
)

resumen_general = pd.DataFrame(
    [{
        "total_reseñas": len(df),
        "score_promedio_sentimiento": score_promedio,
        "total_entidades": int(df["entity_count"].sum()),
        "promedio_entidades_por_reseña": promedio_entidades,
        "tipos_entidad_encontrados": json.dumps(tipos_encontrados, ensure_ascii=False),
    }]
)

print("Cantidad de reseñas y score promedio por etiqueta:")
display(resumen_sentimiento)

print(f"Score promedio de sentimiento: {score_promedio:.4f}")
print(f"Número promedio de entidades por reseña: {promedio_entidades:.2f}")
print("Tipos de entidad encontrados:", ", ".join(tipos_encontrados) or "Ninguno")

print("\nCinco reseñas de menor confianza:")
display(menor_confianza)

print("Menciones por tipo de entidad:")
display(conteo_tipos_entidad)

print("Resumen general:")
display(resumen_general)

### Exportar los resultados
El archivo principal, `laboratorio_4_resultados.csv`, contiene una fila por reseña. Como CSV no admite listas anidadas, la columna `entities` se convierte a texto JSON válido; al volver a leerla se puede reconstruir con `json.loads`.

También se exportan el dataset, las entidades en una tabla independiente y las tablas de estadísticas. Se usa UTF-8 con BOM para facilitar la visualización de tildes en hojas de cálculo. Los archivos se guardan en el directorio de trabajo de Colab.

In [ ]:
df_exportar = df.copy()
df_exportar["entities"] = df_exportar["entities"].apply(
    lambda entidades: json.dumps(entidades, ensure_ascii=False)
)
df_exportar.to_csv(
    "laboratorio_4_resultados.csv", index=False, encoding="utf-8-sig"
)

tablas_adicionales = {
    "laboratorio_4_dataset.csv": dataset,
    "laboratorio_4_entidades.csv": df_entidades,
    "laboratorio_4_resumen_sentimiento.csv": resumen_sentimiento,
    "laboratorio_4_resumen_general.csv": resumen_general,
    "laboratorio_4_menor_confianza.csv": menor_confianza,
    "laboratorio_4_tipos_entidad.csv": conteo_tipos_entidad,
}

for nombre, tabla in tablas_adicionales.items():
    tabla.to_csv(nombre, index=False, encoding="utf-8-sig")

print("Archivos generados:")
for nombre in ["laboratorio_4_resultados.csv", *tablas_adicionales]:
    print("-", nombre)

### Descarga opcional en Google Colab
La siguiente celda descarga el CSV principal. Los demás archivos también estarán disponibles en el panel de archivos de Colab. Cambia `DESCARGAR_CSV` a `True` si deseas activar la descarga.

In [ ]:
DESCARGAR_CSV = False

if DESCARGAR_CSV:
    from google.colab import files
    files.download("laboratorio_4_resultados.csv")

## Referencias
El dataset es simulado; estas fuentes documentan los modelos y la API utilizada:

- [Ficha de RoBERTuito para análisis de sentimientos](https://huggingface.co/pysentimiento/robertuito-sentiment-analysis).
- [Configuración de RoBERTuito](https://huggingface.co/pysentimiento/robertuito-sentiment-analysis/blob/main/config.json).
- [Ficha de BETO ajustado para NER en español](https://huggingface.co/mrm8488/bert-spanish-cased-finetuned-ner).
- [Documentación de pipelines de Transformers](https://huggingface.co/docs/transformers/main_classes/pipelines).

El desarrollo termina en la parte D. El análisis de casos difíciles de la parte E y las preguntas de reflexión quedan fuera del alcance solicitado.